# What is BERT
- Listeneintrag

- [Listeneintrag](https:// Linktext)

- Listeneintrag

- Listeneintrag
- Ref BERT: `Pre-training of Deep Bidirectional Transformers for Language Understanding`

https://arxiv.org/abs/1810.04805

- Understanding searches better then ever before:
https://www.blog.google/products/search/search-language-understanding-bert/

- Good Resource to Read More About the BERT:

http://jalammar.github.io/illustrated-bert/

- Visual Guide to Using BERT:

http://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/

Bidirectional Encoder Representations from Transformers (BERT) is a technique for NLP pre-training developed by Google.

BERT is designed to pretrain deep bidirectional representations from unabeled text by joindly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be finetuned with just one additional output layer to create state-of-the-art-models for a wide range of tasks, such as question aswering and language inference, without substantial taskspesific architecture modifications.

---
There are two existing strategies for applying pre-trained language representations to downstream tasks:

`feature-based and fine-tuning`

The feature-based approach, such as ELMo (Peters et al.,2018a), uses tasks specific architectures that include the pre-trained representations as additional features. The fine-tuning approcah, such as the Generative Pre-Trained Transformer (OpenAI GPT)(Radford et al.,2018), introduces minimal task-specific parameters, and is trained on the downstream tasks by simple fine-tunint all pretrained parameters.

The two approaches share the same objective function during pre-training, where they use undirectional language models to learn general language representations.

# WHY BERT?
- Accurate
- Can be used fo wide variety of task
- Easy to use
- It is game changer in NLP
What is `ktrain`
ktrain is a library to help build, train, debug, and deploy neural networks in the deep learning software framework, keras.

ktrain uses tf.keras in Tensorflow instead of standalone Keras. Inspired by the fastai library, with only a few lines of code, ktrain allows you to easily:

- estimate an optimal learning rate for your model given your data using a learning rate finder
- empoy learning rate schedules such as the triangular learning rate policy, 1cycle policy, and SGDR to more effectively train your model.
- employ fast and easy-to-use-pre-canned models for both text classification (e.g.,NBSVM, fastText,GRU with pretrained word embeddings) and image classification (e.g.,ResNet,Wide Residual Networks,Inception)
- load and preprocess text and image data form a variety of formats
- inspect data points that were misclasified to help improve your model
- leverage a simple prediction API for saving and deploying both models and data-preprocessing steps to make predictions on new raw data.
---
ktrain GitHub: https://github.com/amaiya/ktrain

## Loading and Importing Necessary Libraries

In [ ]:
!pip install transformers
!pip install scikit-learn
!pip install tensorflow
!pip install tf-keras

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

## TPU Detection and Strategy Creation

In [3]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("TPU Active!")
except ValueError:
    print("TPU Not Found, using CPU/GPU.")
    strategy = tf.distribute.get_strategy()  # CPU/GPU strategy

TPU Not Found, using CPU/GPU.


GPU

In [4]:
import tensorflow as tf
print("Count of GPUs:", len(tf.config.list_physical_devices('GPU')))

Count of GPUs: 1


## Loading and Preparing the Dataset

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/refs/heads/master/IMDB-Dataset.csv', dtype=str)

data_length = len(df)
midpoint = data_length // 2

data_train = df.head(midpoint).copy()
data_test = df.tail(data_length - midpoint).copy()

data_train.columns = data_train.columns.str.capitalize()
data_test.columns = data_test.columns.str.capitalize()

data_train.loc[:, 'Sentiment'] = data_train.loc[:, 'Sentiment'].replace({'positive': 1, 'negative': 0})
data_test.loc[:, 'Sentiment'] = data_test.loc[:, 'Sentiment'].replace({'positive': 1, 'negative': 0})

X_train = data_train['Review'].values.astype(str)
y_train = data_train['Sentiment'].values.astype(int)

X_test = data_test['Review'].values.astype(str)
y_test = data_test['Sentiment'].values.astype(int)

print("Training Dataset Size:", data_train.shape)
print("Test Dataset Size:", data_test.shape)

print("\nFirst 5 Rows of Training Dataset:")
print(data_train.head())

print("\nFirst 5 Rows of Test Dataset:")
print(data_test.head())

Training Dataset Size: (25000, 2)
Test Dataset Size: (25000, 2)

First 5 Rows of Training Dataset:
                                              Review Sentiment
0  One of the other reviewers has mentioned that ...         1
1  A wonderful little production. <br /><br />The...         1
2  I thought this was a wonderful way to spend ti...         1
3  Basically there's a family where a little boy ...         0
4  Petter Mattei's "Love in the Time of Money" is...         1

First 5 Rows of Test Dataset:
                                                  Review Sentiment
25000  This movie was bad from the start. The only pu...         0
25001  God, I never felt so insulted in my whole life...         0
25002  Not being a fan of the Coen Brothers or George...         1
25003  The movie Andaz Apna Apna in my books is the t...         1
25004  I have to say I was really looking forward on ...         0


In [6]:
#df = pd.read_csv('https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/refs/heads/master/IMDB-Dataset.csv', dtype=str)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Loading the BERT Tokenizer and Tokenizing the Data

In [7]:
%%time
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_and_align(sentences, labels, max_length):
    tokens = tokenizer(sentences.tolist(),
                      truncation=True,
                      padding='max_length',
                      max_length=max_length,
                      return_tensors='np')
    return tokens['input_ids'], tokens['attention_mask'], np.array(labels)

MAX_LENGTH = 32

train_input_ids, train_attention_masks, train_labels = tokenize_and_align(X_train, y_train, MAX_LENGTH)
test_input_ids, test_attention_masks, test_labels = tokenize_and_align(X_test, y_test, MAX_LENGTH)

print("\nTraining Dataset Token IDs Shape:", train_input_ids.shape)
print("\nTraining Dataset Attention Mask Shape:", train_attention_masks.shape)
print("\nTraining Dataset Label Shape:", train_labels.shape)

print("\nTest Dataset Token IDs Shape:", test_input_ids.shape)
print("\nTest Dataset Attention Mask Shape:", test_attention_masks.shape)
print("\nTest Dataset Label Shape:", test_labels.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]


Training Dataset Token IDs Shape: (25000, 32)

Training Dataset Attention Mask Shape: (25000, 32)

Training Dataset Label Shape: (25000,)

Test Dataset Token IDs Shape: (25000, 32)

Test Dataset Attention Mask Shape: (25000, 32)

Test Dataset Label Shape: (25000,)
CPU times: user 3min 30s, sys: 557 ms, total: 3min 31s
Wall time: 3min 57s


## Loading and Compiling the BERT Model

In [8]:
with strategy.scope():
  bert_model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

  optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
  loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) # We use SparseCategoricalCrossentropy
  metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy') # We use SparseCategoricalAccuracy

  bert_model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

  bert_model.summary()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Training the Model

In [9]:
%%time
EPOCHS = 2
BATCH_SIZE = 32

bert_model.fit(
    [train_input_ids, train_attention_masks],
    train_labels,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=([test_input_ids, test_attention_masks], test_labels)
)

Epoch 1/2
782/782 [==============================] - 303s 330ms/step - loss: 0.5077 - accuracy: 0.7348 - val_loss: 0.4568 - val_accuracy: 0.7690
Epoch 2/2
782/782 [==============================] - 257s 329ms/step - loss: 0.3696 - accuracy: 0.8278 - val_loss: 0.4889 - val_accuracy: 0.7798
CPU times: user 6min 38s, sys: 32.3 s, total: 7min 10s
Wall time: 9min 20s


In [11]:
'''
# @title Saving the Model to Google Drive

from google.colab import drive
drive.mount('/content/drive')

model_save_path = '/content/drive/MyDrive/my_models/bert_sentiment_model'
bert_model.save(model_save_path, save_format='tf')
print(f"Model saved to Google Drive: {model_save_path}")

# Load the saved model from Google Drive
model_save_path = '/content/drive/MyDrive/my_models/bert_sentiment_model'
loaded_model = tf.keras.models.load_model(model_save_path)
print(f"Model loaded from Google Drive: {model_save_path}")
'''

'\n# @title Saving the Model to Google Drive\n\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\nmodel_save_path = \'/content/drive/MyDrive/my_models/bert_sentiment_model\'\nbert_model.save(model_save_path, save_format=\'tf\')\nprint(f"Model saved to Google Drive: {model_save_path}")\n\n# Load the saved model from Google Drive\nmodel_save_path = \'/content/drive/MyDrive/my_models/bert_sentiment_model\'\nloaded_model = tf.keras.models.load_model(model_save_path)\nprint(f"Model loaded from Google Drive: {model_save_path}")\n'

## Evaluating the Model

In [10]:
loss, accuracy = bert_model.evaluate([test_input_ids, test_attention_masks], test_labels, verbose=0)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

y_pred = bert_model.predict([test_input_ids, test_attention_masks])[0]
y_pred_labels = np.argmax(y_pred, axis=1)

print(classification_report(test_labels, y_pred_labels))

Test Loss: 0.48885050415992737
Test Accuracy: 0.7798399925231934
782/782 [==============================] - 64s 78ms/step
              precision    recall  f1-score   support

           0       0.74      0.86      0.80     12474
           1       0.84      0.70      0.76     12526

    accuracy                           0.78     25000
   macro avg       0.79      0.78      0.78     25000
weighted avg       0.79      0.78      0.78     25000



## Making Predictions (Example)

In [12]:
def predict_sentiment(text):
    tokenized_text = tokenizer([text],
                              truncation=True,
                              padding='max_length',
                              max_length=MAX_LENGTH,
                              return_tensors='np')
    output = bert_model.predict([tokenized_text['input_ids'], tokenized_text['attention_mask']])[0]
    predicted_label = np.argmax(output, axis=1)[0]
    return "Positive" if predicted_label == 1 else "Negative"

In [13]:
# Example Usage
example_comment = "This movie was absolutely amazing! The acting was super, and the story was captivating."
prediction = predict_sentiment(example_comment)
print(f"Comment: {example_comment}\nPrediction: {prediction}")

1/1 [==============================] - 0s 93ms/step
Comment: This movie was absolutely amazing! The acting was super, and the story was captivating.
Prediction: Positive


In [16]:
# Example Usage
example_comment2 = "This movie was terrible. I wasted my money."
prediction2 = predict_sentiment(example_comment2)
print(f"Comment: {example_comment2}\nPrediction: {prediction2}")

1/1 [==============================] - 0s 42ms/step
Comment: This movie was terrible. I wasted my money.
Prediction: Negative


In [17]:
# Example Usage
example_comment3 = "The special effects were impressive, but the plot was a bit weak."
prediction3 = predict_sentiment(example_comment3)
print(f"Comment: {example_comment3}\nPrediction: {prediction3}")

1/1 [==============================] - 0s 69ms/step
Comment: The special effects were impressive, but the plot was a bit weak.
Prediction: Negative
